# 02 — Quantum Representation
Feature encoding, ZZFeatureMap circuit, statevector demo.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
SEED = 42
np.random.seed(SEED)
for d in ['../outputs/figures','../outputs/tables','../outputs/kernels',
          '../outputs/logs','../data/processed','../data/raw']:
    os.makedirs(d, exist_ok=True)
print("Setup done.")

Setup done.


## Load & Scale 4 Quantum Features

In [2]:
from src.data import load_wine_quality, scale_features_to_pi, WINE_QUANTUM_FEATURES
df = load_wine_quality('../data/raw/winequality-red.csv')
X_q4 = df[WINE_QUANTUM_FEATURES].values
X_q4_scaled = scale_features_to_pi(X_q4)
print(f"Quantum features: {WINE_QUANTUM_FEATURES}")
print(f"Scale range check: min={X_q4_scaled.min():.4f}, max={X_q4_scaled.max():.4f} (expected [0, π])")
print(f"X_q4_scaled.shape: {X_q4_scaled.shape}")

Loaded wine quality: 1599 rows × 12 columns
Quantum features: ['alcohol', 'volatile acidity', 'sulphates', 'citric acid']
Scale range check: min=0.0000, max=3.1416 (expected [0, π])
X_q4_scaled.shape: (1599, 4)


## ZZFeatureMap Circuit

In [3]:
from src.quantum import build_zz_feature_map
fm = build_zz_feature_map(n_qubits=4, reps=2, entanglement='linear')
# Draw text diagram (Agg backend)
fig = fm.decompose().draw(output='mpl', fold=-1)
fig.savefig('../outputs/figures/07a_zzfeaturemap_circuit.png', dpi=100, bbox_inches='tight')
plt.close()
print("Circuit depth:", fm.decompose().depth())
print("Number of parameters:", fm.num_parameters)
print("Saved circuit diagram.")

ZZFeatureMap: 4 qubits, reps=2, entanglement=linear
  Circuit depth: 19, Parameters: 4


MissingOptionalLibraryError: "The 'pylatexenc' library is required to use 'MatplotlibDrawer'. You can install it with 'pip install pylatexenc'."

## Mathematical Mapping
$$x \rightarrow |\psi(x)\rangle$$
$$S(x_i, x_j) = |\langle\psi(x_i)|\psi(x_j)\rangle|^2$$

In [ ]:
from src.quantum import compute_statevector_demo, get_measurement_probabilities
x0 = X_q4_scaled[0]
sv = compute_statevector_demo(fm, x0)
print(f"Statevector dimension: 2^4 = {len(sv)}")
print(f"Probabilities sum: {np.sum(np.abs(sv)**2):.6f} (should be 1.0)")
probs = get_measurement_probabilities(fm, x0)
print(f"Top-5 measurement probabilities:")
for k, v in sorted(probs.items(), key=lambda x: -x[1])[:5]:
    print(f"  |{k}> : {v:.6f}")

## Demonstrate Fidelity for a Pair

In [ ]:
x1 = X_q4_scaled[1]
from qiskit.quantum_info import Statevector
from src.quantum import build_zz_feature_map
sv0 = Statevector(fm.assign_parameters(x0))
sv1 = Statevector(fm.assign_parameters(x1))
fidelity = abs(sv0.inner(sv1))**2
print(f"S(x0, x1) = |<ψ(x0)|ψ(x1)>|² = {fidelity:.6f}")
print(f"S(x0, x0) = {abs(sv0.inner(sv0))**2:.6f}  (expected ≈ 1.0)")
print("Notebook 02 complete.")